# Reading a real 2 GB ERA5 dataset from IPFS — no local daemon

**CODED / ipfs-agent · companion to [Session 52](https://github.com/esipfed/coded-blog)**

The [companion notebook](./icechunk_on_ipfs_example.ipynb) builds a *tiny* Icechunk repo and reads it back locally. This one is the real thing: it opens an actual **2.08 GB ERA5 2-metre-temperature Icechunk repository** that we've published to IPFS and pinned on a **public gateway** — straight into xarray, over HTTP, with nothing installed but the Python stack.

No IPFS daemon on your machine. No AWS credentials. No auth. Just `pip install` and open a content-addressed dataset by its CID.

> **The dataset:** ERA5 `t2` (2 m air temperature), `time=500 × latitude=721 × longitude=1440`, ~2.08 GB, stored as an [Icechunk](https://icechunk.io) v2 repository and published to IPFS.
>
> **Root CID:** `bafybeicwoya7tmlki3elpgikhgoeswmdrkvcclixvnlgpj4f6xmavptj7y`

## 1. Install dependencies

Just Python packages — no Kubo, no daemon. We read over HTTP from a public IPFS gateway.

In [ ]:
%pip install -q "icechunk>=2.0" "xarray>=2024.0" "zarr>=3" numpy matplotlib

## 2. Open the dataset straight from IPFS

`icechunk.http_storage(base_url)` reads repo objects over HTTP; point it at `<gateway>/ipfs/<CID>` and `xr.open_zarr` turns the session store into a `Dataset`. This is the entire integration — there is no adapter code.

We use our CODED public gateway below. Any IPFS gateway that has the blocks works — e.g. a public one like `https://ipfs.io/ipfs/<CID>` or `https://dweb.link/ipfs/<CID>` (slower, and only while the CID is reachable on the public DHT), or your own local Kubo at `http://127.0.0.1:8080`.

In [ ]:
import icechunk, xarray as xr

GATEWAY = "http://44.239.118.35:8080"   # CODED public IPFS gateway (pinned; see note at bottom)
CID     = "bafybeicwoya7tmlki3elpgikhgoeswmdrkvcclixvnlgpj4f6xmavptj7y"

storage = icechunk.http_storage(f"{GATEWAY}/ipfs/{CID}")   # read-only, over IPFS
repo    = icechunk.Repository.open(storage)
ds      = xr.open_zarr(repo.readonly_session("main").store, consolidated=False)
ds

## 3. It's a normal xarray Dataset — use it normally

Lazy access: only the chunks you touch are fetched over IPFS. Here we grab a single time step (a few chunks, fast).

In [ ]:
import matplotlib.pyplot as plt

t0 = ds.t2.isel(time=0)          # one 721×1440 field
print("this slice:", t0.shape, "· units:", ds.t2.attrs.get("units"))

plt.figure(figsize=(11, 4.5))
plt.pcolormesh(ds.longitude, ds.latitude, t0, shading="auto", cmap="RdBu_r", vmin=230, vmax=310)
plt.colorbar(label="2 m temperature (K)")
plt.title("ERA5 t2 — first time step, read live from IPFS")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.show()

## 4. A full-dataset reduction (streams all 2 GB over IPFS)

The area-weighted global-mean temperature. This pulls every chunk through the gateway — expect ~15–30 s depending on your distance from the node. On our co-located reference it runs ~130–160 MB/s.

In [ ]:
import numpy as np, time

t = time.time()
field = ds.t2.mean("time").values                       # time-mean field (streams full 2 GB)
weights = np.cos(np.deg2rad(ds.latitude.values))          # area weighting
global_mean = float(np.average(field.mean(axis=1), weights=weights))

mb = ds.t2.size * 4 / 1e6
print(f"area-weighted global-mean 2 m temperature = {global_mean:.4f} K")
print(f"read {mb:.0f} MB in {time.time()-t:.1f} s over IPFS")
assert abs(global_mean - 286.5062) < 0.01, "sanity check failed"
print("✅ bit-for-bit match with the reference value (286.5062 K)")

## What just happened

You opened a **2 GB, content-addressed, versioned** climate dataset as xarray **by its CID**, over plain HTTP, with no daemon and no credentials. The bytes are verified by the content hash — if a single byte were wrong, the CID wouldn't match and Icechunk would refuse it.

**Want the exact same dataset from a different source?** Swap `GATEWAY` for any IPFS gateway that has the blocks:
```python
GATEWAY = "https://ipfs.io"      # public DHT (slower, works while the CID is announced)
GATEWAY = "http://127.0.0.1:8080" # your own Kubo, after `ipfs pin add <CID>`
```
The CID never changes — that's the point. Whoever pins it, wherever, serves the identical dataset.

**How it was published** (see the [companion notebook](./icechunk_on_ipfs_example.ipynb) and [Session 52](https://github.com/esipfed/coded-blog)): build an Icechunk repo → `ipfs add -r` → pin the CID on one or more IPFS nodes.

### ⚠️ About this gateway
`44.239.118.35` is a small CODED-run node pinning this one dataset, stood up to make this notebook runnable. **It is temporary** — we plan to tear it down after a few weeks. If the cell above fails to connect, the node is gone; run the self-contained [companion notebook](./icechunk_on_ipfs_example.ipynb) instead (it builds and serves its own repo locally and always works), or re-pin the CID yourself on any IPFS node. Durability on IPFS = *someone* pinning the bytes; a single demo node is not durable by design.